# 07 — Google ADK, LangGraph and Langfuse in depth

### Conversational analytics on BigQuery, with agents that review and validate

Notebook 06 solved the business problems. This notebook shows **what each framework contributes**, using features that each one is genuinely good at. The data is the same dummy e-commerce warehouse in BigQuery (`analytics_agent_dev` is clean; `analytics_agent_corrupted` has planted defects).

| Framework | Problem it solves here | Features used |
|---|---|---|
| **LangGraph** | A1 — *"What exactly did the agent do?"* | the state graph, a diagram of it, step-by-step streaming |
| | A2 — *"No query may run without a human OK"* | **checkpointer** + **interrupt before `execute`**, approve or reject, resume from the checkpoint |
| **Google ADK** | B1 — *"One assistant, several specialists"* | a **multi-agent team**: a coordinator plus a Sales Analyst, a Data Steward and a SQL Auditor, with `transfer_to_agent` hand-offs |
| | B2 — *"Policy must live in code, not prompts"* | `before_tool_callback` guardrail, **session state**, `RunConfig(max_llm_calls)` limit |
| **Langfuse** | C1 — *"Where do time and tokens go?"* | nested traces, generation spans with **model and token usage**, trace URLs |
| | C2 — *"Is answer quality good, and improving?"* | **scores** on traces (verified, reviewed, correct against gold), **sessions** grouping a chat |

Every model call is charged to one **Gemini call budget**, so the free tier (about 20 requests per day per model) is respected. Without Gemini, a clearly labelled deterministic **stand-in** takes the model's place, so every section still runs.

## Architecture

```mermaid
flowchart TB
    U[User] --> CO
    subgraph ADK[Google ADK team - Part B]
        CO[coordinator] -->|transfer_to_agent| SA[sales_analyst]
        CO --> DS[data_steward]
        CO --> AU[sql_auditor]
        GR{{before_tool_callback<br/>guardrail}}
    end
    SA -->|ask_analytics| LG
    DS -->|check_data_quality| DQ[deterministic DQ report<br/>+ reviewer]
    AU -->|audit_sql| VR[validator + SQL & Analytics Reviewer<br/>never executes]
    subgraph LG[LangGraph pipeline - Part A]
        P[precheck] --> PL[plan] --> G[generate_sql] --> V[validate] --> R[review_sql]
        R -->|interrupt_before| H((human<br/>approval)) --> E[execute] --> A[answer] --> VF[verify] --> C[caveats]
    end
    E --> BQ[(BigQuery)]
    DQ --> BQ
    ADK -.spans.-> LF[(Langfuse - Part C<br/>traces, usage, scores, sessions)]
    LG -.spans.-> LF
```

**Division of labour:**

- **ADK** owns the conversation and decides which specialist handles each request.
- **LangGraph** owns the controlled, auditable steps from question to verified answer.
- **Deterministic code** owns everything that must be correct: SQL safety, analytics rules, number verification and data-quality status.
- **Langfuse** records what happened and how good it was.

---
## 1. Setup

### 1.1 Project root and module check

**Expected output:** `All required modules found`, plus the framework versions. If any modules are missing, copy `src/` from the latest zip and restart the kernel.

In [ ]:
# Locate the project root (folder containing pyproject.toml) so the notebook works
# no matter which folder VS Code starts the kernel in, then make `src/` importable.
import os, sys
from pathlib import Path

root = Path.cwd()
while not (root / "pyproject.toml").exists():
    if root.parent == root:
        raise RuntimeError("Could not find pyproject.toml - open the analytics-agent folder in VS Code.")
    root = root.parent
os.chdir(root)
if str(root / "src") not in sys.path:
    sys.path.insert(0, str(root / "src"))
print("Project root:", root)
print("Python:", sys.version.split()[0], "| executable:", sys.executable)

In [ ]:
import importlib.util, importlib.metadata as md
required = ["analytics_agent.bq", "analytics_agent.dq_bigquery", "analytics_agent.period", "analytics_agent.notebook_ui",
            "analytics_agent.agents.analytics_graph", "analytics_agent.agents.adk_team", "analytics_agent.observability"]
missing = [m for m in required if importlib.util.find_spec(m) is None]
if missing:
    raise SystemExit(f"Missing modules {missing}. Copy src/ from the latest analytics-agent.zip into this project and restart the kernel.")
from analytics_agent.agents import analytics_graph as _ag
from analytics_agent import observability as _obs
if not hasattr(_ag, "build_approval_graph") or not hasattr(_obs, "LocalTracer"):
    raise SystemExit("Old versions of analytics_graph.py / observability.py found. Copy src/ from the latest zip and restart the kernel.")
print("All required modules found.")
for pkg in ("google-adk", "langgraph", "langfuse", "google-genai", "google-cloud-bigquery"):
    print(f"  {pkg:<22} {md.version(pkg)}")

In [ ]:
import os, json, asyncio, logging, warnings, datetime as dt
import pandas as pd
from pathlib import Path
from IPython.display import Markdown, display
warnings.filterwarnings("ignore", message=r"\[EXPERIMENTAL\] feature FeatureName.JSON_SCHEMA_FOR_FUNC_DECL")
logging.getLogger("google_adk").setLevel(logging.ERROR)

from analytics_agent.config import load_settings
from analytics_agent.bq import BQSettings, BigQueryRunner, make_client
from analytics_agent.testing import DuckDBRunner
from analytics_agent.data_generation.generator import GenerationConfig
from analytics_agent.dq_bigquery import validate_dataset
from analytics_agent import semantic_layer as sem
from analytics_agent.agents.data_quality_reviewer import DataQualityReviewer
from analytics_agent.agents.analytics_graph import (AnalyticsDeps, run_pipeline, build_analytics_graph, build_approval_graph,
                                                    ask_with_approval, approve, reject)
from analytics_agent.agents.adk_team import AnalyticsTeam, offline_team_models
from analytics_agent.agents.adk_frontend import make_adk_gemini
from analytics_agent.llm import GeminiStructuredClient, LLMError
from analytics_agent.observability import make_tracer, LocalTracer, TeeTracer
from analytics_agent.evaluation import EVAL_SPECS, compare, load_gold
from analytics_agent.notebook_ui import (GeminiBudget, BudgetedLLM, budgeted_adk_model, OfflineStandInModel, CountingRunner,
                                         show_response, format_sql, params_text, is_quota_error)
pd.set_option("display.width", 220); pd.set_option("display.max_colwidth", 110)
AS_OF = GenerationConfig().as_of_ts
print("Imports OK")

### 1.2 ✏️ Settings

| Switch | Meaning |
|---|---|
| `USE_GEMINI` | Use Gemini when the key works; otherwise use the offline stand-in. |
| `GEMINI_CALL_BUDGET` | Hard limit on model calls, shared by LangGraph and every ADK agent. |
| `LOW_QUOTA_MODE` | Deterministic SQL review and data-quality reviewer (fewer calls; all safety checks still run). |
| `TEAM_TURN_LIMIT` | ADK `RunConfig(max_llm_calls=…)` for each team turn. Prevents endless hand-offs. |
| `SCORE_QUESTIONS` | How many gold questions Part C2 scores, at about 3 calls each. |
| `RUN_PART_A/B/C` | Run only some parts, for example Part B on another day, to stay inside a free-tier daily quota. |

**Budget guide** (measured in low-quota mode):

| Part | Approximate calls |
|---|---|
| Part A | about 8 |
| Part B | about 20 (each team turn: coordinator and specialist calls, plus about 3 when the pipeline runs) |
| Part C | about 3 + 3 × `SCORE_QUESTIONS` |
| **Whole notebook** | **about 40** |

On the **free tier** (about 20 requests per day per model) you have three options:

1. Run Parts A and C one day and Part B another day.
2. Give ADK its own model with `ADK_GEMINI_MODEL` in `.env`. Quotas are counted per model, so the team then uses a separate quota.
3. Enable billing and raise `GEMINI_CALL_BUDGET`.

In [ ]:
USE_GEMINI          = True
GEMINI_CALL_BUDGET  = 18
LOW_QUOTA_MODE      = True
TEAM_TURN_LIMIT     = 8
SCORE_QUESTIONS     = 3
RUN_PART_A, RUN_PART_B, RUN_PART_C = True, True, True    # ✏️ e.g. run Part B on another day on the free tier
FORCE_OFFLINE_EMULATOR = False

settings = load_settings(require_project=False)
ADK_MODEL_NAME = os.getenv("ADK_GEMINI_MODEL") or settings.gemini.model
print({k: settings.summary()[k] for k in ("project_id", "location", "gemini_model", "gemini_api_key", "langfuse_enabled")}, "| ADK model:", ADK_MODEL_NAME)

### 1.3 BigQuery (or the offline emulator)

**Expected output:** `BACKEND: bigquery` and your identity, or the **OFFLINE TEST MODE** banner.

In [ ]:
from google.auth.exceptions import DefaultCredentialsError
project = settings.bigquery.project_id
LIVE_BQ = bool(project) and not project.startswith("<") and not FORCE_OFFLINE_EMULATOR
bq_client = None
if LIVE_BQ:
    BQ = BQSettings.from_app_settings(settings)
    try:
        bq_client = make_client(BQ); base_runner = BigQueryRunner(bq_client, BQ)
        who = base_runner.run("SELECT SESSION_USER() AS principal", max_rows=1, dry_run_first=False)
        if not who.ok:
            raise RuntimeError(f"[{who.error_kind}] {who.error_message}")
        print("BACKEND: bigquery |", BQ.project_id, "| identity:", who.rows[0]["principal"])
    except (DefaultCredentialsError, RuntimeError) as e:
        print("BigQuery not usable:", e, "-> offline emulator"); LIVE_BQ = False
if not LIVE_BQ:
    BQ = BQSettings(project_id="offline-emulator"); base_runner = DuckDBRunner(BQ, "data/generated")
    print("=" * 90 + "\nOFFLINE TEST MODE: SQL runs on a local DuckDB emulator of BigQuery, NOT on BigQuery.\n" + "=" * 90)
RUNNER = CountingRunner(base_runner); BACKEND = base_runner.backend
DS = {"clean": BQ.dataset_clean, "corrupted": BQ.dataset_corrupted}

### 1.4 Model (Gemini within a budget, or the stand-in), tracing and data-quality reports

**Tracing setup:**

- `TRACER` sends every span to **Langfuse** when it's enabled, and always to a `LocalTracer` that keeps the same data in memory. You can therefore see the traces in this notebook even before Langfuse is set up.
- ADK's own spans go to Langfuse automatically.

The data-quality reports (deterministic BigQuery SQL, as in notebook 06 Problem 1) feed the Data Steward and the answer caveats.

**Expected output:** the model in use, the tracing status and the data-quality status of both datasets.

In [ ]:
budget = GeminiBudget(GEMINI_CALL_BUDGET)
LIVE_GEMINI = False
if USE_GEMINI and settings.gemini_key_present:
    from typing import Literal
    from pydantic import BaseModel
    class Ping(BaseModel):
        status: Literal["ok"]
    try:
        gemini = GeminiStructuredClient(settings.gemini.model, timeout_s=settings.gemini.timeout_seconds, max_retries=settings.gemini.max_retries)
        BudgetedLLM(gemini, budget).generate(system_instruction="Return JSON only.", prompt='Return status "ok".', schema=Ping, max_output_tokens=256)
        LIVE_GEMINI = True
    except LLMError as e:
        print(f"Gemini not available: {e.kind} - {e.message[:150]}")
if LIVE_GEMINI:
    MODEL = BudgetedLLM(gemini, budget); MODEL_LABEL = f"Gemini {settings.gemini.model}"
    TEAM_MODELS = budgeted_adk_model(make_adk_gemini(ADK_MODEL_NAME), budget)
else:
    budget.limit = 10 ** 9
    MODEL = BudgetedLLM(OfflineStandInModel(), budget); MODEL_LABEL = "offline stand-in (deterministic, NOT Gemini)"
    TEAM_MODELS = {name: budgeted_adk_model(m, budget) for name, m in offline_team_models().items()}

LANGFUSE, msg = make_tracer(settings.langfuse_enabled)
LOCAL = LocalTracer()
TRACER = TeeTracer(LANGFUSE, LOCAL)
print("MODEL:", MODEL_LABEL, "|", msg)

REPORTS = {v: validate_dataset(RUNNER, ds, AS_OF, dataset_label=v) for v, ds in DS.items()}
print({v: f"{r.overall_status} ({r.counts()['fail']} failing checks)" for v, r in REPORTS.items()})

def make_deps(variant: str, llm=None) -> AnalyticsDeps:
    return AnalyticsDeps(llm=llm or MODEL, runner=RUNNER, settings=BQ, dataset=DS[variant], tracer=TRACER,
                         dq_report_provider=lambda v=variant: REPORTS[v],
                         dq_reviewer=DataQualityReviewer(MODEL if (LIVE_GEMINI and not LOW_QUOTA_MODE) else None),
                         use_llm_sql_reviewer=LIVE_GEMINI and not LOW_QUOTA_MODE)

def calls_since(n0: int) -> str:
    return f"{budget.used - n0} model calls" + (f" (budget left: {budget.remaining})" if LIVE_GEMINI else "")

---
## Part A — LangGraph: a controlled, observable workflow

### A1. *"What exactly did the agent do?"* — the graph and step-by-step streaming

**Problem.** A black-box agent can't be audited. When an answer looks wrong, you need to see which step went wrong.

**LangGraph features:**

- the pipeline **is** a `StateGraph`: nodes are plain Python functions, and edges are routing rules written in code;
- `get_graph().draw_mermaid()` renders the real graph, generated from the code rather than drawn by hand;
- `graph.stream(..., stream_mode="updates")` emits each node's result **as it completes**, so you can watch the question move through precheck, plan, SQL, validation, review, execution, answer, verification and caveats.

**Model calls:** about 3.

In [ ]:
if RUN_PART_A:
    budget.current = "A1"
    graph = build_analytics_graph(make_deps("clean"))
    display(Markdown("**The compiled graph (generated from the code):**\n\n```mermaid\n" + graph.get_graph().draw_mermaid() + "\n```"))
else:
    print("Skipped (RUN_PART_A = False).")

In [ ]:
if RUN_PART_A:
    budget.current = "A1"
    n0 = budget.used
    question = "What was our monthly net sales over the last six months?"
    print(f"Streaming: {question}\n")
    final = None
    for update in graph.stream({"question": question, "history": [], "steps": []}, stream_mode="updates"):
        node, values = next(iter(update.items()))
        last = (values.get("steps") or [{}])[-1] if isinstance(values, dict) else {}
        detail = {k: v for k, v in last.items() if k in ("intent", "metrics", "period", "period_source", "ok", "approved",
                                                         "status", "row_count", "source", "count", "errors", "issues") and v not in (None, [], "")}
        print(f"  ✓ {node:<13} {detail}")
        if node == "finalize":
            final = values["response"]
    show_response(final, model_label=MODEL_LABEL, show_sql=False)
    print(calls_since(n0))
else:
    print("Skipped (RUN_PART_A = False).")

**How to read it.** Each line is one node finishing:

- `plan` shows the metric and the period, with "last 6 complete months" resolved in code;
- `validate` and `review_sql` show the safety and analytics checks;
- `execute` shows the BigQuery result;
- `verify` shows whether every number in the answer was found in the rows.

If a check rejects the SQL, you'll see `generate_sql` appear again: that's LangGraph's bounded retry edge.

### A2. *"No query may run without a human OK"* — checkpointer and interrupt

**Problem.** In some teams, no automatically generated SQL may run against production data until a person has seen it: for cost, compliance or confidence.

**LangGraph features:**

- `compile(checkpointer=..., interrupt_before=["execute"])` makes the graph **pause** after validation and review but **before any query runs**, saving its full state in a checkpoint;
- **approve** calls `graph.invoke(None, config)`, which **resumes from the checkpoint** and continues with execute, answer, verify and caveats;
- **reject** calls `graph.update_state(..., as_node="finalize")`, which records the decision and ends the run without executing anything.

Each question gets its own `thread_id`. The checkpointer explicitly registers the project's state types, so newer LangGraph versions won't block them.

**Model calls:** about 6 (two questions; the rejected one never reaches answer generation).

In [ ]:
if RUN_PART_A:
    budget.current = "A2"
    n0, q0 = budget.used, RUNNER.runs
    approval_graph = build_approval_graph(make_deps("clean"))

    def show_pending(p):
        if not p.awaiting_approval:
            display(Markdown(f"**Finished without needing approval:** `{p.finished.status}` — {p.finished.answer}")); return
        display(Markdown(f"#### ⏸ Awaiting approval: {p.question}\n\n- **Period:** {p.period} · **metrics:** {p.metrics} · **tables:** {list(p.tables)}\n"
                         f"- **Automated checks:** validator ✓ · SQL & Analytics Reviewer: {'approved' if p.review['approved'] else p.review['issues']}\n\n"
                         f"```sql\n{format_sql(p.sql)}\n```\n\n**Parameters:** {params_text(p.parameters)}"))

    pending_ok = ask_with_approval("What was our monthly net sales over the last six months?", make_deps("clean"), approval_graph)
    show_pending(pending_ok)
    print("Queries executed while waiting for approval:", RUNNER.runs - q0)
else:
    print("Skipped (RUN_PART_A = False).")

The human decision is made here. **Approve** the first query: the graph resumes from its checkpoint and finishes.

In [ ]:
if RUN_PART_A:
    budget.current = "A2"
    q1 = RUNNER.runs
    approved = approve(approval_graph, pending_ok, make_deps("clean"))
    show_response(approved, model_label=MODEL_LABEL, show_sql=False)
    print("Queries executed after approval:", RUNNER.runs - q1)
else:
    print("Skipped (RUN_PART_A = False).")

**Reject** a second query. The decision is written into the checkpoint and nothing is executed.

In [ ]:
if RUN_PART_A:
    budget.current = "A2"
    pending_no = ask_with_approval("Which product categories generate the most sales?", make_deps("clean"), approval_graph)
    show_pending(pending_no)
    q2 = RUNNER.runs
    rejected = reject(approval_graph, pending_no, "A category breakdown is not needed for this report.")
    display(Markdown(f"**Result:** `{rejected.status}` — {rejected.answer}\n\nLast step recorded: `{rejected.steps[-1]}`"))
    print("Queries executed after rejection:", RUNNER.runs - q2, "|", calls_since(n0))
else:
    print("Skipped (RUN_PART_A = False).")

**How to read it.**

- **Queries executed while waiting: 0.** The graph stopped before `execute`.
- **After approval: 1.** Exactly the query that was approved ran.
- **After rejection: 0,** and the checkpoint records a `human_review` step with your reason.

The same mechanism could route expensive queries (based on the dry-run byte estimate) to a human automatically.

---
## Part B — Google ADK: a team of specialist agents

### B1. *"One assistant, several specialists"* — multi-agent delegation

**Problem.** Users ask very different things: sales figures, whether the data can be trusted, whether *their own* SQL is right. One prompt doing everything becomes vague and hard to control.

**ADK features:**

- `LlmAgent(sub_agents=[...])`: a **coordinator** that never answers itself, but delegates with ADK's built-in `transfer_to_agent`;
- three specialists, each with only the tools it needs:

| Agent | Tool | What the tool does |
|---|---|---|
| `sales_analyst` | `ask_analytics`, `list_metrics` | runs the LangGraph pipeline from Part A |
| `data_steward` | `check_data_quality` | the deterministic data-quality report plus the Data Quality Reviewer |
| `sql_auditor` | `audit_sql` | the SQL validator plus the SQL & Analytics Reviewer, on SQL **pasted by an analyst**; it **never executes** |

- `InMemorySessionService` keeps the conversation. ADK continues with the active specialist, and specialists can hand off to their peers.
- After every turn, the final reply is **verified against the tool outputs**; unsupported replies are replaced by the verified tool answer.

**Model calls:** about 2–3 ADK calls per turn, plus about 3 when the pipeline runs.

In [ ]:
if RUN_PART_B:
    budget.current = "B1"
    n0 = budget.used
    team = AnalyticsTeam({v: make_deps(v) for v in DS}, TEAM_MODELS, tracer=TRACER, max_llm_calls_per_turn=TEAM_TURN_LIMIT,
                         dq_reviewer=DataQualityReviewer(None if LOW_QUOTA_MODE or not LIVE_GEMINI else MODEL))
    team_session = await team.new_session(user_id="analyst-1")
    analyst_sql = (f"SELECT SUM(o.order_total) AS total_sales FROM `{BQ.project_id}.{DS['clean']}.orders` o "
                   f"JOIN `{BQ.project_id}.{DS['clean']}.order_items` li ON li.order_id = o.order_id "
                   f"WHERE o.order_status IN ('processing', 'shipped', 'delivered')")

    async def team_turn(text):
        before = budget.used
        try:
            with TRACER.session(session_id=team_session, user_id="analyst-1", tags=["notebook-07", "adk-team"]):
                t = await team.ask(text, team_session, user_id="analyst-1")
        except Exception as exc:  # noqa: BLE001
            if is_quota_error(exc):
                display(Markdown(f"**Stopped: Gemini quota/budget exhausted** at *{text[:80]}*. Earlier answers are complete and verified."))
                return None
            raise
        display(Markdown(f"**USER:** {text}\n\n**ASSISTANT:** {t.answer}\n\n"
                         f"- route: {' → '.join(t.route)} · tools: `{[c['tool'] for c in t.tool_calls]}`"
                         + (f" · **blocked:** {[b['reason'] for b in t.blocked_calls]}" if t.blocked_calls else "")
                         + f"\n- verified against tool outputs: {t.verified} · fallback: {t.fallback_used} · {budget.used - before} model calls"))
        return t

    for text in ["What were monthly net sales in the first half of 2026?",
                 "And which product category contributed the most in that period?",
                 "Can I trust these numbers? Check the corrupted dataset.",
                 f"Please audit this SQL before I put it in a dashboard: {analyst_sql}"]:
        if await team_turn(text) is None:
            break
    print(calls_since(n0))
else:
    print("Skipped (RUN_PART_B = False).")

**How to read it.**

1. **Sales question:** the route is `coordinator → sales_analyst`, which calls the pipeline.
2. **Follow-up:** ADK stays with the `sales_analyst`, which turns the follow-up into a standalone question for the same period.
3. **Trust question:** the analyst **hands over to a peer**, the `data_steward`, which reports "not fit for use" for the corrupted data.
4. **Pasted SQL:** this goes to the `sql_auditor`, whose verdict is **NEEDS CHANGES (join fan-out)**. `SUM(order_total)` after joining order lines counts each order once per line. The query was never executed.

### B2. *"Policy must live in code, not in prompts"* — callbacks, session state, turn limits

**Problem.** An instruction such as "only use the approved datasets" can be ignored, or talked around. Policies need enforcement **in code**, at the point where the action happens.

**ADK features:**

- **`before_tool_callback`** runs before *every* tool call of every specialist. Returning a dict **skips the tool** and uses that dict as the result. The guardrail enforces the dataset allowlist and size limits.
- **Session state** (`tool_context.state`) persists across turns. The guardrail counts tool calls and blocked calls there.
- **`RunConfig(max_llm_calls=TEAM_TURN_LIMIT)`** caps each turn, so agents can't hand work back and forth forever or burn your quota.

The request below asks for an unapproved dataset. The tool must **never run**.

**Model calls:** about 3.

In [ ]:
if RUN_PART_B:
    budget.current = "B2"
    n0, q0 = budget.used, RUNNER.runs
    await team_turn("Show total net sales from the staging dataset.")
    print("Queries executed for that request:", RUNNER.runs - q0)
    print("ADK session state:", await team.state(team_session, user_id="analyst-1"))
    print(calls_since(n0))
else:
    print("Skipped (RUN_PART_B = False).")

**How to read it.**

- **Blocked:** "dataset 'staging' is not approved". The callback stopped the call before the tool ran, so **0 queries** were executed.
- **Session state:** `tool_calls` counts every call made in this session, and `blocked_tool_calls` counts the refusals. ADK keeps this state across turns, so it could also drive rate limits or audits.

---
## Part C — Langfuse: traces, cost and quality

**Enable Langfuse** (free cloud account, or self-hosted):

1. Create a project at cloud.langfuse.com. Then **Settings → API keys**.
2. Add the keys to `.env`, then restart the kernel:

```dotenv
ENABLE_LANGFUSE=true
LANGFUSE_PUBLIC_KEY=pk-lf-...
LANGFUSE_SECRET_KEY=sk-lf-...
LANGFUSE_BASE_URL=https://cloud.langfuse.com
```

Payloads pass through a mask that redacts emails and key-like strings and truncates long row lists. Without Langfuse, the same data is shown from the `LocalTracer`.

### C1. *"Where do time and tokens go?"* — nested traces with model usage

**Langfuse features:**

- one **trace** per question, containing nested spans of different types: `guardrail`, `generation`, `evaluator`, `tool` and `agent`;
- **generation** spans carry the **model name and token usage** (input, output and reasoning tokens) reported by Gemini, which Langfuse turns into cost;
- `get_trace_url` gives a direct link to the trace.

**Model calls:** about 3.

In [ ]:
if RUN_PART_C:
    budget.current = "C1"
    n0 = budget.used
    with TRACER.session(session_id="notebook-07-c1", user_id="analyst-1", tags=["notebook-07"]):
        resp = run_pipeline("What percentage of orders were delivered late, by carrier?", [], make_deps("clean"))
    show_response(resp, model_label=MODEL_LABEL, show_sql=False, show_rows=False)
    spans = LOCAL.spans_frame(LOCAL.spans[-1]["trace_id"]) if LOCAL.spans else pd.DataFrame()
    cols = [c for c in ("name", "type", "model", "input_tokens", "output_tokens", "seconds", "status", "ok", "approved") if c in spans.columns]
    display(Markdown("**Spans recorded for this question** (identical to what Langfuse receives):")); display(spans[cols])
    url = TRACER.trace_url(resp.trace_id)
    print("Langfuse trace:", url or "(Langfuse not enabled — table above shows the local copy)")
    print(calls_since(n0))
else:
    print("Skipped (RUN_PART_C = False).")

**How to read it.** Indented names show nesting. `generation` rows are model calls. With Gemini they show the model and its token counts; with the stand-in they show the stand-in's label and no tokens.

In Langfuse, open the trace link to see the same tree with inputs, outputs, timings and cost. **BigQuery job IDs** are on the `execute_query` span, so you can cross-check them in BigQuery's Job history.

### C2. *"Is answer quality good, and improving?"* — scores and sessions

**Langfuse features:**

- **scores** (`create_score`) attach measurements to a trace, as booleans, numbers or categories. Here every answer gets:
  - `verified` — every number was found in the rows;
  - `sql_review_approved`;
  - `answer_source` — Gemini verified, or deterministic fallback;
  - **`correct_vs_gold`** — the result rows match an independently computed gold answer;
- **sessions** (`propagate_attributes(session_id=...)`) group related traces. The ADK chat in Part B is one session; this evaluation run is another.

Over time, Langfuse's dashboards show whether these scores move as prompts, models or the semantic layer change.

**Model calls:** about 3 × `SCORE_QUESTIONS`.

In [ ]:
if RUN_PART_C:
    budget.current = "C2"
    n0 = budget.used
    gold = load_gold("data/manifests/expected_results_clean.json")
    eval_session = f"eval-{dt.datetime.now():%Y%m%d-%H%M%S}"
    rows = []
    with TRACER.session(session_id=eval_session, user_id="evaluator", tags=["notebook-07", "gold-eval"]):
        for spec in EVAL_SPECS[:SCORE_QUESTIONS]:
            try:
                r = run_pipeline(spec.question, [], make_deps("clean"))
            except LLMError as e:
                print("Stopped:", e.kind); break
            correct = compare(spec, gold[spec.test_id]["expected_result"], r.rows)[0] if r.rows else False
            scores = {"verified": int(bool(r.verification.get("ok"))) if r.verification else 0,
                      "sql_review_approved": int(bool(r.review.get("sql_review_approved"))) if r.review else 0,
                      "correct_vs_gold": int(correct)}
            for name, value in scores.items():
                TRACER.score(name, value, trace_id=r.trace_id, session_id=eval_session, data_type="BOOLEAN")
            TRACER.score("answer_source", r.answer_source or "none", trace_id=r.trace_id, session_id=eval_session, data_type="CATEGORICAL")
            rows.append({"case": spec.test_id, "question": spec.question, **scores, "answer_source": r.answer_source, "trace": r.trace_id})
    TRACER.flush()
    scored = pd.DataFrame(rows); display(scored)
    if len(scored):
        print(f"correct_vs_gold: {scored.correct_vs_gold.mean():.0%} · verified: {scored.verified.mean():.0%} · session: {eval_session} · {calls_since(n0)}")
    display(Markdown("**All scores recorded in this notebook run:**")); display(LOCAL.scores_frame())
else:
    print("Skipped (RUN_PART_C = False).")

**How to read it.** Each row is one traced question with its scores.

- In Langfuse, filter by **session** (`eval-…`) to see this evaluation run, or open **Scores** to chart `correct_vs_gold` across runs.
- With the stand-in, the scores show that the pipeline works end to end. With Gemini, they measure real answer quality.
- The ADK chat from Part B appears as **one session**, with every turn and agent hand-off inside it.

---
## Summary

In [ ]:
print(f"Model: {MODEL_LABEL} | backend: {BACKEND} | model calls used: {budget.used} | queries run: {RUNNER.runs}")
print("Model calls per section:", budget.by_problem)
print(f"Traces recorded: {LOCAL.spans_frame().trace_id.nunique() if LOCAL.spans else 0} | scores: {len(LOCAL.scores)} | Langfuse: {'enabled' if LANGFUSE.enabled else 'disabled'}")
TRACER.flush()

| Framework | What it does best | What we used it for |
|---|---|---|
| **LangGraph** | Explicit, testable, resumable workflows | A pipeline where no step can be skipped; streaming; **human approval** with checkpoints, resume and reject |
| **Google ADK** | Conversational, tool-using, multi-agent systems | A **coordinator with three specialists**, peer hand-offs, a guardrail **callback**, **session state**, a per-turn **call limit** |
| **Langfuse** | Seeing and measuring LLM applications | Nested **traces** with model and token usage, **scores** (verified, reviewed, correct against gold), **sessions** |
| **Deterministic code** | Being right every time | SQL safety, analytics rules, number verification, data-quality status: Gemini never decides these |

### Troubleshooting

| Symptom | Fix |
|---|---|
| `Old versions … found` in 1.1 | Copy `src/` from the latest zip, then restart the kernel. |
| `quota` / `RESOURCE_EXHAUSTED` | Free-tier limit. Raise `GEMINI_CALL_BUDGET` only with billing; set `ADK_GEMINI_MODEL` to another model (quotas are per model); lower `SCORE_QUESTIONS`. |
| ADK error about too many model calls | `TEAM_TURN_LIMIT` stopped a turn on purpose (endless hand-offs). Rephrase, or raise the limit slightly. |
| No traces in Langfuse | Check the four `.env` values and restart the kernel. 1.4 must say `Langfuse enabled`. Run the summary cell, which flushes. |
| `Deserializing unregistered type` warning | You're using an older `analytics_graph.py`; the new checkpointer registers all state types. |
| Team answers show `fallback: True` | ADK's wording claimed something the tools didn't return; the verified tool answer was shown instead. |

### References

- Google ADK, multi-agent systems and callbacks: https://google.github.io/adk-docs/
- LangGraph persistence and human-in-the-loop: https://langchain-ai.github.io/langgraph/
- Langfuse, tracing, scores and sessions: https://langfuse.com/docs